<a href="https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

# Store the token in DuckDB's session variable
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

# Configure Hugging Face authentication
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

print("DuckDB connected successfully.")
print("HF token loaded:", HF_TOKEN is not None)

DuckDB connected successfully.
HF token loaded: True


In [ ]:
FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

FEB = f"{FACT}/month=2026-02/*.parquet"

print("Feature window: February 2026")
print("Warehouse path configured.")

Feature window: February 2026
Warehouse path configured.


In [ ]:
test = con.sql(f"""
SELECT COUNT(*) AS rows
FROM read_parquet('{FEB}')
""").df()

print(test)

      rows
0  7355108


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distribution check

I will inspect the February 2026 distributions of organic impressions, organic clicks, average search position, and engagement seconds per pageview.

Traffic-related fields may be heavy-tailed, so I will compare the median with higher percentiles instead of relying only on the mean. This gives a more careful view of the observed content population.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-06 Step 1: Distribution check

import pandas as pd
import numpy as np

# Load February 2026 data
dist_df = con.sql(f"""
SELECT
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_total_engagement_sec
FROM read_parquet('{FEB}')
""").df()

# Convert numeric fields safely
numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_total_engagement_sec"
]

for col in numeric_cols:
    dist_df[col] = pd.to_numeric(
        dist_df[col],
        errors="coerce"
    )

# One row per content item
dist_df = (
    dist_df
    .groupby("content_hash_id", as_index=False)
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_pageviews=("ga4_pageviews", "sum"),
        ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum")
    )
)

# Safe engagement metric
dist_df["engagement_sec_per_pageview"] = np.where(
    dist_df["ga4_pageviews"] > 0,
    dist_df["ga4_total_engagement_sec"] /
    dist_df["ga4_pageviews"],
    np.nan
)

fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "engagement_sec_per_pageview"
]

print("February 2026 distribution summary")
print("=" * 65)

for col in fields:

    s = dist_df[col].dropna()

    print(f"\n{col}")
    print(f"n       : {len(s)}")
    print(f"min     : {s.min():.4f}")
    print(f"median  : {s.median():.4f}")
    print(f"p75     : {s.quantile(0.75):.4f}")
    print(f"p90     : {s.quantile(0.90):.4f}")
    print(f"p95     : {s.quantile(0.95):.4f}")
    print(f"max     : {s.max():.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February 2026 distribution summary

gsc_impressions
n       : 321546
min     : 0.0000
median  : 0.0000
p75     : 99.0000
p90     : 1059.0000
p95     : 2626.7500
max     : 203401.0000

gsc_clicks
n       : 321546
min     : 0.0000
median  : 0.0000
p75     : 0.0000
p90     : 2.0000
p95     : 7.0000
max     : 3310.0000

gsc_avg_position
n       : 153559
min     : 0.0000
median  : 7.9354
p75     : 14.8066
p90     : 29.1929
p95     : 42.2013
max     : 633.0000

ga4_pageviews
n       : 321546
min     : 0.0000
median  : 0.0000
p75     : 0.0000
p90     : 1.0000
p95     : 3.0000
max     : 7792.0000

engagement_sec_per_pageview
n       : 33365
min     : 0.0000
median  : 0.0000
p75     : 1.0769
p90     : 7.2190
p95     : 20.8571
max     : 1104.0000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal tests

I will test three observable relationships using February 2026 data.

1. **Average search position → CTR:** checks whether CTR changes across search-position buckets. This is linked to FlyRank's CTR-vs-position logic.

2. **Impression volume → CTR:** checks whether higher-volume content shows a consistent CTR pattern. This tests the volume signal used in quick-win reasoning.

3. **Engagement seconds per pageview → CTR:** checks whether observed engagement levels are associated with different CTR levels. This is treated as a directional signal, not proof of causation.

Each test will be assigned CONFIRMED, OPPOSITE, MIXED, or FALSE based only on the observed bucket pattern.

In [ ]:
# ML-06 Step 2: Three signal tests

signal_df = dist_df.copy()

# Calculate CTR safely
signal_df["ctr"] = np.where(
    signal_df["gsc_impressions"] > 0,
    signal_df["gsc_clicks"] / signal_df["gsc_impressions"],
    np.nan
)

signal_df = signal_df[
    signal_df["gsc_impressions"] > 0
].copy()

print("Rows available for signal tests:", len(signal_df))


# ==========================================================
# SIGNAL 1 — Average position vs CTR
# ==========================================================

signal_df["position_bucket"] = pd.cut(
    signal_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    include_lowest=True
)

position_test = (
    signal_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 1 — Average Position vs CTR")
print(position_test.to_string(index=False))

values = position_test["mean_ctr"].dropna().tolist()

if len(values) >= 2:
    if all(values[i] >= values[i+1]
           for i in range(len(values)-1)):
        signal1_verdict = "CONFIRMED"
    elif all(values[i] <= values[i+1]
             for i in range(len(values)-1)):
        signal1_verdict = "OPPOSITE"
    else:
        signal1_verdict = "MIXED"
else:
    signal1_verdict = "FALSE"

print("Signal 1 verdict:", signal1_verdict)


# ==========================================================
# SIGNAL 2 — Impression volume vs CTR
# ==========================================================

signal_df["volume_bucket"] = pd.cut(
    signal_df["gsc_impressions"],
    bins=[0, 100, 1000, 10000, np.inf],
    labels=["<100", "100-999", "1K-9,999", "10K+"],
    include_lowest=True
)

volume_test = (
    signal_df
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        mean_ctr=("ctr", "mean"),
        mean_position=("gsc_avg_position", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — Impression Volume vs CTR")
print(volume_test.to_string(index=False))

values = volume_test["mean_ctr"].dropna().tolist()

if len(values) >= 2:
    if all(values[i] >= values[i+1]
           for i in range(len(values)-1)):
        signal2_verdict = "CONFIRMED"
    elif all(values[i] <= values[i+1]
             for i in range(len(values)-1)):
        signal2_verdict = "OPPOSITE"
    else:
        signal2_verdict = "MIXED"
else:
    signal2_verdict = "FALSE"

print("Signal 2 verdict:", signal2_verdict)


# ==========================================================
# SIGNAL 3 — Engagement per pageview vs CTR
# ==========================================================

eng_df = signal_df[
    signal_df["engagement_sec_per_pageview"].notna()
].copy()

if len(eng_df) >= 4:

    # Rank first so qcut always has unique values
    eng_df["engagement_rank"] = (
        eng_df["engagement_sec_per_pageview"]
        .rank(method="first")
    )

    eng_df["engagement_bucket"] = pd.qcut(
        eng_df["engagement_rank"],
        q=4,
        labels=[
            "Q1 lowest",
            "Q2",
            "Q3",
            "Q4 highest"
        ]
    )

    engagement_test = (
        eng_df
        .groupby("engagement_bucket", observed=False)
        .agg(
            n=("content_hash_id", "count"),
            mean_ctr=("ctr", "mean")
        )
        .reset_index()
    )

    print("\nSIGNAL 3 — Engagement per Pageview vs CTR")
    print(engagement_test.to_string(index=False))

    values = engagement_test["mean_ctr"].dropna().tolist()

    if len(values) >= 2:
        if all(values[i] <= values[i+1]
               for i in range(len(values)-1)):
            signal3_verdict = "CONFIRMED"
        elif all(values[i] >= values[i+1]
                 for i in range(len(values)-1)):
            signal3_verdict = "OPPOSITE"
        else:
            signal3_verdict = "MIXED"
    else:
        signal3_verdict = "FALSE"

else:
    print("\nSIGNAL 3 — Not enough valid observations.")
    signal3_verdict = "FALSE"

print("Signal 3 verdict:", signal3_verdict)

Rows available for signal tests: 153559

SIGNAL 1 — Average Position vs CTR
position_bucket     n  mean_ctr
            1-3 19243  0.010598
           4-10 75898  0.005195
          11-20 31700  0.003059
            20+ 26718  0.002509
Signal 1 verdict: CONFIRMED

SIGNAL 2 — Impression Volume vs CTR
volume_bucket     n  mean_ctr  mean_position
         <100 73435  0.007411      14.723334
      100-999 46837  0.002429      12.383713
     1K-9,999 29957  0.003097       8.759707
         10K+  3330  0.003433       8.078404
Signal 2 verdict: MIXED

SIGNAL 3 — Engagement per Pageview vs CTR
engagement_bucket    n  mean_ctr
        Q1 lowest 6150  0.004475
               Q2 6149  0.004759
               Q3 6149  0.005712
       Q4 highest 6149  0.007659
Signal 3 verdict: CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag-linked signal: Average position vs CTR

This test is linked to FlyRank's CTR-vs-position / CTR-fix logic. The assumption is that pages with worse average search positions tend to receive lower CTR. In the February 2026 observed data, mean CTR decreases across the position buckets from 0.010598 for positions 1–3 to 0.002509 for positions 20+. This supports the directional assumption behind the flag.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Flag-linked test
# FlyRank flag: CTR-vs-position / CTR-fix logic

flag_test = position_test.copy()

print("FLAG-LINKED TEST — CTR vs Average Position")
print(flag_test.to_string(index=False))

# Compare best-position bucket with worst-position bucket
if len(flag_test) >= 2:
    best_ctr = flag_test["mean_ctr"].iloc[0]
    worst_ctr = flag_test["mean_ctr"].iloc[-1]

    print("\nBest-position bucket mean CTR:", round(best_ctr, 6))
    print("Worst-position bucket mean CTR:", round(worst_ctr, 6))

    if best_ctr > worst_ctr:
        flag_verdict = "CONFIRMED"
    elif best_ctr < worst_ctr:
        flag_verdict = "OPPOSITE"
    else:
        flag_verdict = "MIXED"
else:
    flag_verdict = "FALSE"

print("Flag-linked verdict:", flag_verdict)

FLAG-LINKED TEST — CTR vs Average Position
position_bucket     n  mean_ctr
            1-3 19243  0.010598
           4-10 75898  0.005195
          11-20 31700  0.003059
            20+ 26718  0.002509

Best-position bucket mean CTR: 0.010598
Worst-position bucket mean CTR: 0.002509
Flag-linked verdict: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The observed February 2026 data supports the directional relationship between search position and CTR: CTR was highest for positions 1–3 and lowest for positions 20+. The content team can use position and engagement signals as decision-support for prioritizing review, while treating the signals as directional rather than proof that a refresh will improve performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.